# Add translations to Swedish questions

In [1]:
import json

In [2]:
questions_file = "../data/swe_Latn/swe_Latn_test_dev_long_doc_subset_annotated_splitted_paragraphs_Qwen3.5-122B-A10B-GPTQ-Int4_generate_questions.jsonl"
translation_file = "../data/swe_Latn/swe_Latn_test_dev_long_doc_subset_annotated_splitted_paragraphs_Qwen3_translations.jsonl"

In [3]:
with open(questions_file) as q, open(translation_file) as t:
    t_lines = t.readlines()
    q_lines = q.readlines()

translations = [json.loads(l) for l in t_lines]
questions = [json.loads(l) for l in q_lines]


In [4]:
translations[0]

{'original_question': '# Question:\nHur många tecken bör rubriktaggarna innehålla för att undvika att de blir avklippta?',
 'tranlation_prompt': '<|im_start|>system\nYou are a helpful and focused AI assistant.\n\n    Always follow the user’s instructions carefully and complete the requested tasks to the best of your ability.\n    Provide clear, accurate, and relevant responses that stay on topic.\n    Do not include extra information or content that was not requested by the user.<|im_end|>\n<|im_start|>user\nTask:\nTranslate the given questions from Swedish to English.\nRules:\n- The questions meaning should not be changed\n- Translation should be in English\n\nOutput format:\n# Question:\n<the translated question>\n\nInput question:\n<<<question\n# Question:\nHur många tecken bör rubriktaggarna innehålla för att undvika att de blir avklippta?\nquestion\n>>><|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n',
 'translated_question': '# Question:\nHow many characters should titl

In [5]:
updated_questions = []
for t,q in zip(translations,questions):
    if t['original_question'] == q['generated_text']:
        new_dict = q.copy()
        new_dict['translation']=t['translated_question']
        updated_questions.append(new_dict)

assert len(updated_questions)==len(questions)

In [6]:
updated_questions[0]

{'original_paragraph': '- Titel tagg\n- Sökmotorvänliga URL\n- Metabeskrivning\n- Titeln lockar användarna in på sidorna från sökmotorn och du bör inkludera sökord i titeln. Kolla igenom och se till att alla sidor har en titel och metabeskrivning. Med verktyg som Screaming Frog kan du hämta hem alla dina sidor och se hur titlar och metabeskrivningar är satta.\n- Det räcker inte med att bara skriva en titel utan content managers måste lägga kärlek på orden och hur dessa är skrivna. Det är viktigt att skapa unika och korrekta titlar som lockar till klick och ökar CTR:en (click through rate) och som använder maximalt utrymme.\n- Se till att rubriktaggarna består av 55-59 tecken har du fler riskerar de bli avklippta.\n- Använd nyckelordet du vill synas på i det första 100 orden\n- Metabeskrivningar är ett smart sätt att visa företagets unique selling points dvs fördelarna med att anlita företaget eller köpa produkterna och locka besökare att klicka på länkarna som pekar mot sajten',
 'prom

In [7]:
out_file = "../data/swe_Latn/swe_Latn_test_dev_long_doc_subset_annotated_splitted_paragraphs_Qwen3.5-122B-A10B-GPTQ-Int4_generate_questions_with_translations.jsonl"
with open(out_file,"w") as f:
    for l in updated_questions:
        j_l = json.dumps(l,ensure_ascii=False)
        f.write(j_l + '\n')

# Parse questions, documents, and paragraph starts and ends together into single files

In [8]:
import json

In [9]:
def read_jsonl(path):
    out = []
    with open(path, "r", encoding="utf-8") as f:
        for ln, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                out.append(obj)
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON on line {ln} of {path}: {e}") from e
                continue
    return out

In [10]:
fin_questions_path = "../data/fin_Latn/fin_Latn_test_dev_long_doc_subset_annotated_splitted_paragraphs_Qwen3.5-122B-A10B-GPTQ-Int4_generate_questions_annotated.jsonl"
swe_questions_path = "../data/swe_Latn/swe_Latn_test_dev_long_doc_subset_annotated_splitted_paragraphs_Qwen3.5-122B-A10B-GPTQ-Int4_generate_questions_with_translations_annotated.jsonl"
fin_docs_path = "../data/fin_Latn/fin_Latn_test_dev_long_doc_subset_annotated_splitted_paragraphs.jsonl"
swe_docs_path = "../data/swe_Latn/swe_Latn_test_dev_long_doc_subset_annotated_splitted_paragraphs.jsonl"

In [11]:
fin_q = read_jsonl(fin_questions_path)
swe_q = read_jsonl(swe_questions_path)
fin_d = read_jsonl(fin_docs_path)
swe_d = read_jsonl(swe_docs_path)

## Filter bad questions

In [12]:
fin_q = [q for q in fin_q if q['good_question']==1]
swe_q = [q for q in swe_q if q['good_question']==1]

In [13]:
print(len(fin_q))
print(len(swe_q))

909
905


## Filter the source docs based on good questions

In [14]:
fin_q_ids = set([q['id'] for q in fin_q])
swe_q_ids = set([q['id'] for q in swe_q])

In [15]:
print(len(fin_d))
print(len(swe_d))

1016
1018


In [16]:
fin_d = [d for d in fin_d if d['id'] in fin_q_ids]
swe_d = [d for d in swe_d if d['id'] in swe_q_ids]

In [17]:
print(len(fin_d))
print(len(swe_d))

909
905


## Combine the data

In [18]:
import pandas as pd

In [19]:
print(fin_d[0].keys())
print(fin_q[0].keys())

dict_keys(['f', 'o', 's', 'rs', 'u', 'c', 'ts', 'de', 'crawl_id', 'lang', 'prob', 'text', 'xml', 'html_lang', 'cluster_size', 'seg_langs', 'id', 'filter', 'pii', 'doc_scores', 'web-register', 'good_text', 'paragraphs'])
dict_keys(['original_paragraph', 'prompt', 'generated_text', 'id', 'good_question'])


In [20]:
swe_data = pd.DataFrame(swe_d)
swe_queries = pd.DataFrame(swe_q)
swe_merged = swe_data.merge(swe_queries, on="id", how="inner")

In [21]:
fin_data = pd.DataFrame(fin_d)
fin_queries = pd.DataFrame(fin_q)
fin_merged = fin_data.merge(fin_queries, on="id", how="inner")

In [22]:
swe_merged = swe_merged.drop(columns=['good_text', 'paragraphs','prompt','translation','good_question'])

In [23]:
fin_merged = fin_merged.drop(columns=['good_text', 'paragraphs','prompt','good_question'])

In [24]:
swe_merged.rename(columns={'generated_text': 'query'},inplace=True)
fin_merged.rename(columns={'generated_text': 'query'},inplace=True)

## Add start, end, and sim

In [25]:
from rapidfuzz import fuzz

In [26]:
def fuzzy_find(doc,para,min_score=95):
    """
    Finds the best fuzzy substring match of para in doc.
    Returns (start, end, score) or None.
    """
    if not doc or not para:
        return None, None, None

    alignment = fuzz.partial_ratio_alignment(doc, para)
    if alignment.score < min_score:
        print(alignment.score)
        print(para)
        print()
        print(doc[alignment.src_start:alignment.src_end])
        return None,None,None
        
    start = alignment.src_start
    end = alignment.src_end
    return (start,end,alignment.score)

In [27]:
fin_merged[["paragraph_char_start", "paragraph_char_end", "paragraph_similarity_to_original"]] = fin_merged.apply(
    lambda row: pd.Series(fuzzy_find(row["text"], row["original_paragraph"])),
    axis=1)

83.41058794828007
15 Valaistussuunnittelu Suunnitteluvinkkejä L med < 500 cd/m² Näkökentässä olevien seinäpintojen luminanssin suhde työtason luminanssiin tulisi olla suurempi kuin 1:5! Tilan pintojen valaistussuhteet Tilan pintojen valaistusvoimakkuuden tulisi olla oikeassa suhteessa työtason valaistusvoimakkuuteen, jotta standardin luminanssisuhteiden vaatimukset täyttyvät. Esimerkiksi pienluminanssi- ja downlight-valaisimia käytettäessä on riski, että seinien yläosat ja katto jäävät liian tummiksi. Standardissa EN on annettu suositukset vain tilan pintojen heijastussuhteille ja ne on esitetty alla olevassa taulukossa. Tilan seinäpintojen luminanssin tulisi olla vähintään 30 cd/m 2, kun pyritään hyvään näkömukavuuteen. Koska valaistussuunnittelu perustuu käytännössä valaistusvoimakkuuksien laskentaan, on tapana ilmoittaa pintojen luminanssit suhteellisten valaistusvoimakkuuksien avulla. Standardi ei ota kantaa tilan pintojen valaistusvoimakkuuteen, joten apuna voidaan käyttää oheisen

In [28]:
swe_merged[["paragraph_char_start", "paragraph_char_end", "paragraph_similarity_to_original"]] = swe_merged.apply(
    lambda row: pd.Series(fuzzy_find(row["text"], row["original_paragraph"])),
    axis=1)

79.73372781065089
Naturlig familjeplanering - afrikanska boskapsskötare förstår vad det handlar om!
Varför riskera livet för några få procents ökning av graviditetsskyddet? Det finns ändå inga preventivmedel som ger 100 % graviditetsskydd.Teoretiskt sett ger kondomer ett bättre smittskydd. Men kondomernas smittskyddande effekt uppvägs ju lätt av den livsstil som kondomkampanjerna uppmanar till. Det är bättre att avstå från tillfälliga förbindelser och vänta med samlag efter partnerbyte tills båda har testat sig mot hiv. Undrar för resten hur kampanjmakarna har tänkt sig att barn ska bli gjorda med 100 % smittskydd? Som för övrigt inte kan uppnås med kondomer, utan endast genom celibat.

istiska partiet. Budskapet är tydligt: Svenskarna, och särskilt de svenska folkpartisterna, vet bättre än alla andra folkslag på jorden.
Naturlig familjeplanering - afrikanska boskapsskötare förstår vad det handlar om!
Teoretiskt sett ger kondomer ett bättre smittskydd. Men kondomernas smittskyddande ef

In [29]:
#drop the documents where XML and parsed text are too different
fin_merged = fin_merged.dropna(subset=["paragraph_char_start"])
swe_merged = swe_merged.dropna(subset=["paragraph_char_start"])

In [30]:
fin_merged[["paragraph_char_start", "paragraph_char_end"]] = fin_merged[["paragraph_char_start", "paragraph_char_end"]].astype("int64")
swe_merged[["paragraph_char_start", "paragraph_char_end"]] = swe_merged[["paragraph_char_start", "paragraph_char_end"]].astype("int64")

In [31]:
import sys
import unicodedata
# Following https://github.com/google-research/bert/blob/master/tokenization.py
_punct_chars = set([
    chr(i) for i in range(sys.maxunicode)
    if (unicodedata.category(chr(i)).startswith('P') or
        ((i >= 33 and i <= 47) or (i >= 58 and i <= 64) or
         (i >= 91 and i <= 96) or (i >= 123 and i <= 126)))
])

_cjk_chars = set([
    chr(i) for i in range(sys.maxunicode)
    if ((i >= 0x4E00 and i <= 0x9FFF) or
        (i >= 0x3400 and i <= 0x4DBF) or
        (i >= 0x20000 and i <= 0x2A6DF) or
        (i >= 0x2A700 and i <= 0x2B73F) or
        (i >= 0x2B740 and i <= 0x2B81F) or
        (i >= 0x2B820 and i <= 0x2CEAF) or
        (i >= 0xF900 and i <= 0xFAFF) or
        (i >= 0x2F800 and i <= 0x2FA1F))
])

_strip_chars = set([
    chr(i) for i in range(sys.maxunicode)
    if i == 0 or i == 0xfffd or (
            chr(i) not in '\t\n\r' and unicodedata.category(chr(i)) in ('Cc', 'Cf')
    )
])


def _mapto(char):
    if char in _punct_chars or char in _cjk_chars:
        # Add space around punctuation and CJK chars for string.split()
        return ' ' + char + ' '
    elif char in _strip_chars:
        return None
    else:
        raise ValueError(char)


_translation_table = str.maketrans({
    c: _mapto(c) for c in _punct_chars | _cjk_chars | _strip_chars
})

def basic_tokenize(text):
    return text.translate(_translation_table).split()

In [32]:
def tokens_before_paragraph(char_index,text):
    t = text[:char_index]
    return len(basic_tokenize(t))

In [33]:
fin_merged["tokens_before_paragraph"] = fin_merged.apply(
    lambda row: pd.Series(tokens_before_paragraph(row["paragraph_char_start"], row["text"])),
    axis=1)

In [34]:
swe_merged["tokens_before_paragraph"] = swe_merged.apply(
    lambda row: pd.Series(tokens_before_paragraph(row["paragraph_char_start"], row["text"])),
    axis=1)

In [35]:
fin_merged['tokens_before_paragraph'].describe()

count      903.000000
mean      4784.655592
std       6231.730311
min          0.000000
25%       1229.500000
50%       2884.000000
75%       6239.000000
max      56430.000000
Name: tokens_before_paragraph, dtype: float64

In [36]:
fin_merged['tokens_before_paragraph'].median()

np.float64(2884.0)

In [37]:
fin_merged["tokens_before_paragraph"].quantile(0.12)

np.float64(542.24)

In [38]:
fin_merged["tokens_before_paragraph"].quantile(0.23)

np.float64(1078.76)

In [39]:
swe_merged['tokens_before_paragraph'].describe()

count       900.000000
mean       5462.856667
std        9689.572917
min           0.000000
25%        1257.750000
50%        3028.000000
75%        6172.250000
max      120351.000000
Name: tokens_before_paragraph, dtype: float64

In [40]:
swe_merged['tokens_before_paragraph'].median()

np.float64(3028.0)

In [41]:
swe_merged['tokens_before_paragraph'].quantile(0.12)

np.float64(555.3199999999999)

In [42]:
swe_merged['tokens_before_paragraph'].quantile(0.22)

np.float64(1084.58)

# Format the data to follow MLDR
- Three types of files where each must contain at least dev and test splits
1. Queries:
    - id
    - text
2. References:
    - query-id
    - corpus-id
    - score 
3. Corpus:
    - id
    - text
## References

In [43]:
from sklearn.model_selection import train_test_split

In [44]:
def add_query_id(row):
    return f"query-{row['id']}"

In [45]:
swe_merged['query-id']=swe_merged.apply(add_query_id,axis=1)
fin_merged['query-id']=fin_merged.apply(add_query_id,axis=1)

In [46]:
swe_qrels = swe_merged[['query-id','id']]
fin_qrels = fin_merged[['query-id','id']]

In [47]:
swe_qrels['score']=1
swe_qrels.rename(columns={'id': 'corpus-id'},inplace=True)
fin_qrels['score']=1
fin_qrels.rename(columns={'id': 'corpus-id'},inplace=True)

/tmp/reunamo1/6369220/ipykernel_2113832/3446446355.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  swe_qrels['score']=1
/tmp/reunamo1/6369220/ipykernel_2113832/3446446355.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  swe_qrels.rename(columns={'id': 'corpus-id'},inplace=True)
/tmp/reunamo1/6369220/ipykernel_2113832/3446446355.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/s

In [48]:
swe_dev_qrels, swe_test_qrels= train_test_split(swe_qrels, test_size=0.75, random_state=66)
fin_dev_qrels, fin_test_qrels= train_test_split(fin_qrels, test_size=0.75, random_state=66)

In [49]:
print(len(swe_dev_qrels))
print(len(swe_test_qrels))
print(len(fin_dev_qrels))
print(len(fin_test_qrels))

225
675
225
678


In [50]:
fin_dev_qrels.to_json("/scratch/project_2018556/finnish-swedish-long-document-retrieval/results/fin-qrels/dev.jsonl", orient="records", lines=True,force_ascii=False)
fin_test_qrels.to_json("/scratch/project_2018556/finnish-swedish-long-document-retrieval/results/fin-qrels/test.jsonl", orient="records", lines=True,force_ascii=False)

In [51]:
swe_dev_qrels.to_json("/scratch/project_2018556/finnish-swedish-long-document-retrieval/results/swe-qrels/dev.jsonl", orient="records", lines=True,force_ascii=False)
swe_test_qrels.to_json("/scratch/project_2018556/finnish-swedish-long-document-retrieval/results/swe-qrels/test.jsonl", orient="records", lines=True,force_ascii=False)

## Queries

In [52]:
fin_queries = fin_merged[['query-id','query']]
swe_queries = swe_merged[['query-id','query']]

In [53]:
fin_queries.rename(columns={'query-id': 'id','query':'text'},inplace=True)
swe_queries.rename(columns={'query-id': 'id','query':'text'},inplace=True)

/tmp/reunamo1/6369220/ipykernel_2113832/3459627730.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fin_queries.rename(columns={'query-id': 'id','query':'text'},inplace=True)
/tmp/reunamo1/6369220/ipykernel_2113832/3459627730.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  swe_queries.rename(columns={'query-id': 'id','query':'text'},inplace=True)


In [54]:
import re
def remove_question_str(t):
    s = re.sub(r'^\s*#\s*Question:\s*\n?', '',t)
    return s

In [55]:
fin_queries['text'] = fin_queries['text'].apply(remove_question_str)
swe_queries['text'] = swe_queries['text'].apply(remove_question_str)

/tmp/reunamo1/6369220/ipykernel_2113832/2510427098.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fin_queries['text'] = fin_queries['text'].apply(remove_question_str)
/tmp/reunamo1/6369220/ipykernel_2113832/2510427098.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  swe_queries['text'] = swe_queries['text'].apply(remove_question_str)


In [56]:
fin_test_ids = set(fin_test_qrels['query-id'])
swe_test_ids = set(swe_test_qrels['query-id'])

swe_test_mask = swe_queries["id"].isin(swe_test_ids)
fin_test_mask = fin_queries["id"].isin(fin_test_ids)

fin_test_queries  = fin_queries[fin_test_mask].copy()       # rows with id in ids
fin_dev_queries  = fin_queries[~fin_test_mask].copy()       # rows with  not id in ids

swe_test_queries  = swe_queries[swe_test_mask].copy()       # rows with id in ids
swe_dev_queries  = swe_queries[~swe_test_mask].copy()      # rows with  not id in ids

In [57]:
# query meta data
fin_test_meta  = fin_merged[fin_test_mask].copy()       # rows with id in ids
fin_dev_meta  = fin_merged[~fin_test_mask].copy()       # rows with  not id in ids

swe_test_meta  = swe_merged[swe_test_mask].copy()       # rows with id in ids
swe_dev_meta  = swe_merged[~swe_test_mask].copy()

In [58]:
fin_test_meta = fin_test_meta[["query-id", "original_paragraph","paragraph_char_start","paragraph_char_end","paragraph_similarity_to_original","tokens_before_paragraph"]]
fin_dev_meta = fin_dev_meta[["query-id", "original_paragraph","paragraph_char_start","paragraph_char_end","paragraph_similarity_to_original","tokens_before_paragraph"]]
swe_test_meta = swe_test_meta[["query-id", "original_paragraph","paragraph_char_start","paragraph_char_end","paragraph_similarity_to_original","tokens_before_paragraph"]]
swe_dev_meta = swe_dev_meta[["query-id", "original_paragraph","paragraph_char_start","paragraph_char_end","paragraph_similarity_to_original","tokens_before_paragraph"]]

In [59]:
swe_test_meta.to_json("/scratch/project_2018556/finnish-swedish-long-document-retrieval/results/swe-queries-meta/test.jsonl", orient="records", lines=True,force_ascii=False)
swe_dev_meta.to_json("/scratch/project_2018556/finnish-swedish-long-document-retrieval/results/swe-queries-meta/dev.jsonl", orient="records", lines=True,force_ascii=False)

fin_test_meta.to_json("/scratch/project_2018556/finnish-swedish-long-document-retrieval/results/fin-queries-meta/test.jsonl", orient="records", lines=True,force_ascii=False)
fin_dev_meta.to_json("/scratch/project_2018556/finnish-swedish-long-document-retrieval/results/fin-queries-meta/dev.jsonl", orient="records", lines=True,force_ascii=False)


In [60]:
fin_dev_queries.to_json("/scratch/project_2018556/finnish-swedish-long-document-retrieval/results/fin-queries/dev.jsonl", orient="records", lines=True,force_ascii=False)
fin_test_queries.to_json("/scratch/project_2018556/finnish-swedish-long-document-retrieval/results/fin-queries/test.jsonl", orient="records", lines=True,force_ascii=False)

In [61]:
swe_dev_queries.to_json("/scratch/project_2018556/finnish-swedish-long-document-retrieval/results/swe-queries/dev.jsonl", orient="records", lines=True,force_ascii=False)
swe_test_queries.to_json("/scratch/project_2018556/finnish-swedish-long-document-retrieval/results/swe-queries/test.jsonl", orient="records", lines=True,force_ascii=False)

## Corpus

In [62]:
fin_corpus_path = "../data/fin_Latn/fin_Latn_100k_long_doc_corpus.jsonl"
swe_corpus_path = "../data/swe_Latn/swe_Latn_100k_long_doc_corpus.jsonl"

In [63]:
import shutil

def make_corpus_data(data_path):
    #read data
    if "swe" in data_path:
        lang="swe"
    if "fin" in data_path:
        lang="fin"
    out_path = f"/scratch/project_2018556/finnish-swedish-long-document-retrieval/results/{lang}-corpus/test.jsonl"
    out_path_meta = f"/scratch/project_2018556/finnish-swedish-long-document-retrieval/results/{lang}-corpus-meta/test.jsonl"
    with open(data_path, "r", encoding="utf-8") as f_in, open(out_path_meta,"w") as meta_out, open(out_path,"w") as corp_out:
        for ln, line in enumerate(f_in, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                corp = {'id':obj['id'],'text':obj['text']}
                corp_out.write(json.dumps(corp, ensure_ascii=False) + "\n")
                corp_meta = {k:v for k,v in obj.items() if k not in ['text','xml']}
                meta_out.write(json.dumps(corp_meta, ensure_ascii=False) + "\n")

            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON on line {ln} of {path}: {e}") from e
                continue
    shutil.copy(f"/scratch/project_2018556/finnish-swedish-long-document-retrieval/results/{lang}-corpus/test.jsonl",f"/scratch/project_2018556/finnish-swedish-long-document-retrieval/results/{lang}-corpus/dev.jsonl") 
    shutil.copy(f"/scratch/project_2018556/finnish-swedish-long-document-retrieval/results/{lang}-corpus-meta/test.jsonl",f"/scratch/project_2018556/finnish-swedish-long-document-retrieval/results/{lang}-corpus-meta/dev.jsonl") 


In [64]:
make_corpus_data(fin_corpus_path)

In [65]:
make_corpus_data(swe_corpus_path)

In [66]:
#get the min max dates from corpus
def get_date_range(corpus_meta_data):
    # if your column is not already datetime
    df = pd.read_json(corpus_meta_data, lines=True)
    df['ts'] = pd.to_datetime(df['ts'])
    min_date = df['ts'].min()
    max_date = df['ts'].max()
    return min_date, max_date
    

In [67]:
swe_min,swe_max=get_date_range("/scratch/project_2018556/finnish-swedish-long-document-retrieval/results/swe-corpus-meta/test.jsonl")
fin_min,fin_max=get_date_range("/scratch/project_2018556/finnish-swedish-long-document-retrieval/results/fin-corpus-meta/test.jsonl")
print(f"swe min: {swe_min},swe max: {swe_max}")
print(f"fin min: {fin_min},swe max {fin_max}")

swe min: 2011-04-14 22:21:35+00:00,swe max: 2024-12-15 00:01:25+00:00
fin min: 2011-03-18 07:23:00+00:00,swe max 2024-12-14 23:38:44+00:00
